In [ ]:
!pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
!pip install pandas openai


In [ ]:
!pip install groq
from groq import Groq
import os

os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
import pandas as pd
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))




In [ ]:
from google.colab import files
import pandas as pd

# Upload CSV file
uploaded = files.upload()
# Gets the uploaded file name
file_name = list(uploaded.keys())[0]

# Loads the uploaded file into pandas DataFrame
# Using pd.read_excel because the uploaded file has an .xlsx extension
df = pd.read_excel(file_name)

# Displays first few rows
df.head()

Saving first try.csv.xlsx to first try.csv (3).xlsx


,Unnamed: 0,NAME OF THE STUDENT,UNIVERSITY,PROGRAM NAME,Specialisation,SEMESTER,Domain,GENERAL MANAGEMENT SCORE (OUT of 50),Domain Specific SCORE (OUT 50),TOTAL SCORE (OUT of 100),RANK,PERCENTILE
0,0,Camila Wood,"Stanford University, USA",B.Com,Honours,5th,Finance,50,50,100.0,1,1.000000
1,1,Alexander Thompson,"Stanford University, USA",B.Com,Financial Services,5th,Finance,50,50,100.0,2,0.993377
2,2,Liam Taylor,"Harvard University, USA",B.Com,Accounting Analytics,5th,BA,50,50,100.0,3,0.986755
3,3,Evelyn Jenkins,"Stanford University, USA",B.Com,Honours,5th,Finance,49,50,99.0,4,0.980132
4,4,Michael Jackson,"Harvard University, USA",MBA,International Business,3rd,IB,50,49,99.0,5,NaN


In [ ]:
# 1. Remove unnecessary index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# 2. Drop rows where critical score values are missing
clean_df = df.dropna(subset=[
    "GENERAL MANAGEMENT SCORE (OUT of 50)",
    "Domain Specific SCORE (OUT 50)"
])

# 3. Convert score columns to numeric (safety step)
clean_df["GENERAL MANAGEMENT SCORE (OUT of 50)"] = pd.to_numeric(
    clean_df["GENERAL MANAGEMENT SCORE (OUT of 50)"]
)

clean_df["Domain Specific SCORE (OUT 50)"] = pd.to_numeric(
    clean_df["Domain Specific SCORE (OUT 50)"]
)

# 4. Recalculate TOTAL SCORE to show transformation logic
clean_df["CALCULATED TOTAL SCORE"] = (
    clean_df["GENERAL MANAGEMENT SCORE (OUT of 50)"] +
    clean_df["Domain Specific SCORE (OUT 50)"]
)

# 5. Create a performance category (business logic)
def performance_label(score):
    if score >= 90:
        return "Excellent"
    elif score >= 75:
        return "Good"
    else:
        return "Average"

clean_df["PERFORMANCE CATEGORY"] = clean_df["CALCULATED TOTAL SCORE"].apply(performance_label)

# Show final cleaned data
clean_df.head()


,NAME OF THE STUDENT,UNIVERSITY,PROGRAM NAME,Specialisation,SEMESTER,Domain,GENERAL MANAGEMENT SCORE (OUT of 50),Domain Specific SCORE (OUT 50),TOTAL SCORE (OUT of 100),RANK,PERCENTILE,CALCULATED TOTAL SCORE,PERFORMANCE CATEGORY
0,Camila Wood,"Stanford University, USA",B.Com,Honours,5th,Finance,50,50,100.0,1,1.000000,100,Excellent
1,Alexander Thompson,"Stanford University, USA",B.Com,Financial Services,5th,Finance,50,50,100.0,2,0.993377,100,Excellent
2,Liam Taylor,"Harvard University, USA",B.Com,Accounting Analytics,5th,BA,50,50,100.0,3,0.986755,100,Excellent
3,Evelyn Jenkins,"Stanford University, USA",B.Com,Honours,5th,Finance,49,50,99.0,4,0.980132,99,Excellent
4,Michael Jackson,"Harvard University, USA",MBA,International Business,3rd,IB,50,49,99.0,5,NaN,99,Excellent


In [ ]:
pipeline_code = f"""
Dataset Description:
The dataset contains student academic evaluation details with the following columns:
{', '.join(df.columns)}

Pipeline Logic:
1. Load the CSV dataset containing student evaluation details.
2. Remove rows where critical score columns are missing:
   - GENERAL MANAGEMENT SCORE (OUT of 50)
   - Domain Specific SCORE (OUT 50)
3. Convert score columns to numeric format.
4. Recalculate total score as:
   GENERAL MANAGEMENT SCORE (OUT of 50) + Domain Specific SCORE (OUT 50)
5. Create a new column called PERFORMANCE CATEGORY:
   - Excellent: total score >= 90
   - Good: total score >= 75
   - Average: total score < 75
6. Output the cleaned and transformed dataset.
"""
print(pipeline_code)



Dataset Description:
The dataset contains student academic evaluation details with the following columns:
NAME OF THE STUDENT, UNIVERSITY, PROGRAM NAME, Specialisation, SEMESTER, Domain, GENERAL MANAGEMENT SCORE (OUT of 50), Domain Specific SCORE (OUT 50), TOTAL SCORE (OUT of 100), RANK, PERCENTILE

Pipeline Logic:
1. Load the CSV dataset containing student evaluation details.
2. Remove rows where critical score columns are missing:
   - GENERAL MANAGEMENT SCORE (OUT of 50)
   - Domain Specific SCORE (OUT 50)
3. Convert score columns to numeric format.
4. Recalculate total score as:
   GENERAL MANAGEMENT SCORE (OUT of 50) + Domain Specific SCORE (OUT 50)
5. Create a new column called PERFORMANCE CATEGORY:
   - Excellent: total score >= 90
   - Good: total score >= 75
   - Average: total score < 75
6. Output the cleaned and transformed dataset.



In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"


In [ ]:
prompt = f"""
You are a data engineer.
Generate technical documentation for the following pipeline:

{pipeline_code}

Include:
- Purpose
- Input data
- Transformation logic
- Output
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

documentation = response.choices[0].message.content
print(documentation)


**Student Academic Evaluation Pipeline Documentation**

**Purpose**

This pipeline is designed to load, clean, and transform student academic evaluation data from a CSV file. The pipeline will remove rows with missing score columns, convert score columns to numeric format, recalculate total score, and create a new column for performance category.

**Input Data**

* **Dataset:** student_evaluation_data.csv
* **Columns:**
	+ NAME OF THE STUDENT
	+ UNIVERSITY
	+ PROGRAM NAME
	+ Specialisation
	+ SEMESTER
	+ Domain
	+ GENERAL MANAGEMENT SCORE (OUT of 50)
	+ Domain Specific SCORE (OUT 50)
	+ TOTAL SCORE (OUT of 100)
	+ RANK
	+ PERCENTILE

**Transformation Logic**

1. **Load CSV Dataset**
	* Read the CSV file using pandas: `pd.read_csv('student_evaluation_data.csv')`
2. **Remove Rows with Missing Critical Score Columns**
	* Filter out rows where either `GENERAL MANAGEMENT SCORE (OUT of 50)` or `Domain Specific SCORE (OUT 50)` is missing using pandas `dropna` function:
	```python
dataset = da

In [ ]:
question = "Explain the transformation logic of this pipeline in simple terms.in telugu under 10 lines"

chat_prompt = f"""
You are a helpful data engineer assistant.

Here is the pipeline documentation:
{documentation}

Answer the following question clearly and concisely:
{question}
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": chat_prompt}
    ]
)

answer = response.choices[0].message.content
print(answer)


ఈ పైపులైన్ తరంగం లాగా మార్చు దిద్దుబాటు జ్ఞానం.

1. మనం ఇన్‌పుట్ ఫైల్‌ని చదవాలి. (తెలుగు నంబర్లు మరియు నామమాత్రం సంఖ్యలు వేరు చెబిస్తుంది.)
2. మిగిలిన సంభావ్య నకరాత్మక విలువలను నిష్క్రమించండి.
3. వివరణాత్మక గుణకాలను సంఖ్య రూపంలోకి మార్చండి.
4. మొత్తం గుణకం కోసం కొత్త వర్గం లను కలిపి.
5. చివరగా, సాధన విభాగంలో జ్యామితిలో అర్హతలు కల్పించండి.


In [ ]:
runbook_prompt = f"""
You are a data engineer responsible for operating this data pipeline.

Based on the pipeline described below, generate an operational runbook.

Pipeline description:
{pipeline_code}

The runbook must include:
1. Prerequisites (software, libraries, files needed)
2. Steps to run the pipeline
3. Common failure scenarios
4. How to troubleshoot and recover from failures

Do not invent columns or tools that are not mentioned.
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": runbook_prompt}
    ]
)

runbook = response.choices[0].message.content
print(runbook)


**Operational Runbook: Student Academic Evaluation Pipeline**

**Version:** 1.0

**Prerequisites:**

1. **Software:** Python 3.9 or later installed on the machine.
2. **Libraries:**
	* `pandas` for data manipulation and analysis.
	* `numpy` for numerical computations.
3. **Files:**
	* `student_evaluation.csv`: the dataset containing student academic evaluation details.
	* `transformations.py`: a Python script containing the pipeline logic.

**Steps to Run the Pipeline:**

1. **Load prerequisites:** Ensure Python, pandas, and numpy are installed. Also, verify that `student_evaluation.csv` is in the correct location.
2. **Run the pipeline:** Execute the `transformations.py` script using Python: `python transformations.py`
3. **Monitor pipeline execution:** Observe the output of the pipeline, which should be stored in a transformed dataset.

**transformations.py:**
```python
import pandas as pd
import numpy as np

# Load dataset
def load_dataset(file_path):
    return pd.read_csv(file_pat